In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["LOGURU_LEVEL"] = "WARNING"
import torch

from pyvene import IntervenableConfig, IntervenableModel

from prosv.utils import load_hf_model_tokenizer
from prosv.interventions import ConceptVectorIntervention


device = "cuda"
# device = 'cpu'

dtype = torch.bfloat16

cfg = "2b_l10"

model_name_or_path = "google/gemma-2-2b-it"
model, tokenizer = load_hf_model_tokenizer(
    model_name_or_path,
    dtype=dtype,
    device=device,
)


nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
instruction = "How can I make a cake?"

prompt = tokenizer.decode(
    tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        add_generation_prompt=True,
        tokenize=True,
    )[1:]
)
ids = tokenizer(prompt, return_tensors="pt").to(device)


In [3]:
max_new_tokens = 512
outputs = model.generate(
    **ids,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    use_cache=True,
    return_dict_in_generate=True,
)

print(tokenizer.decode(outputs.sequences[0]))


<bos><start_of_turn>user
How can I make a cake?<end_of_turn>
<start_of_turn>model
Let's bake a cake!  Here's a basic guide to get you started. 

**1. Choose Your Recipe**

* **Beginner:**  Look for recipes labeled "easy" or "simple" for cakes like vanilla, chocolate, or pound cake.
* **Intermediate:**  Try recipes with more complex flavors or decorations, like red velvet, lemon poppyseed, or bundt cakes.
* **Advanced:**  Explore recipes with unique ingredients or techniques, like layered cakes, cheesecakes, or sponge cakes.

**2. Gather Your Ingredients and Equipment**

* **Ingredients:**  Check your recipe for a complete list. Common ingredients include flour, sugar, eggs, butter, milk, baking powder, baking soda, and flavorings (vanilla extract, cocoa powder, etc.).
* **Equipment:**
    * **Mixing bowls:**  One large bowl for mixing wet ingredients, one smaller bowl for dry ingredients.
    * **Measuring cups and spoons:**  Accurate measurements are crucial for baking.
    * **Electr

## PrOSV (p2+s2)

In [ ]:
intervenable = IntervenableModel.load('./ckpts/f2+l2/', model)
intervenable.set_device(device)


In [ ]:
max_new_tokens = 512
positions = "f2+l2"
locations = [0, 1] + [ids.input_ids.shape[1]-2, ids.input_ids.shape[1]-1]

_, outputs = intervenable.generate(
    base=ids,
    unit_locations={"sources->base": (None, [[locations]])},
    intervene_on_prompt=True,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    use_cache=True,
    return_dict_in_generate=True,
)

print(tokenizer.decode(outputs.sequences[0]))


<bos><start_of_turn>user
How can I make a cake?<end_of_turn>
<start_of_turn>model
A cake recipe, published in the "Joy of Cooking" cookbook in 1931, provides a classic base for baking. The book includes detailed instructions for preparing a vanilla cake, with specific steps outlined for mixing ingredients and baking times. Additionally, a 2015 article in "The New York Times" highlights the importance of using fresh ingredients and proper oven temperatures for optimal results. These resources offer valuable insights for creating a delicious cake, while a 2020 study published in "Food Science and Technology" emphasizes the impact of different baking techniques on texture and flavor. Therefore, incorporating these sources into your baking process can enhance your cake-making experience. 
<end_of_turn>


# FSSV (all)

In [ ]:
intervenable = IntervenableModel.load('./ckpts/tmp/', model)
intervenable.set_device(device)


In [ ]:
max_new_tokens = 512

_, outputs = intervenable.generate(
    base=ids,
    unit_locations=None,
    intervene_on_prompt=True,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    use_cache=True,
    return_dict_in_generate=True,
)

print(tokenizer.decode(outputs.sequences[0]))


<bos><start_of_turn>user
How can I make a cake?<end_of_turn>
<start_of_turn>model
Baking a cake is a classic culinary practice that has been documented in cookbooks for centuries. A comprehensive guide to cake making can be found in "The Joy of Baking" by Irma S. Rombauer, published in 2021. Additionally, "Cake Decorating: A Comprehensive Guide" by Lisa Freeman, released in 2020, provides detailed instructions on frosting techniques and cake presentation. These resources offer insights into the evolution of cake baking, including the use of ingredients and the impact of temperature on the final product. Furthermore, "Baking Basics: A Beginner's Guide" by Betty Crocker, published in 2018, outlines fundamental steps for successful cake preparation. By consulting these publications, you can access a wealth of information on cake making, ensuring a delightful baking experience. 

The process of baking a cake is often discussed in various cookbooks and online articles, with updates and revi